# Раздел 3 · Коллекции и точки: модель данных Qdrant

In [ ]:
%pip install qdrant-client sentence-transformers ipykernel --quiet


In [ ]:
import json
import subprocess
import numpy as np
import polars as pl
from qdrant_client import models
from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
)

print("OK: библиотеки импортированы")


/home/dimon/vscode/examples_qdrant/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK: библиотеки импортированы


### Docker: тот же сервер, что и в остальных ноутбуках курса

In [2]:
CONTAINER_NAME = "qdrant_lecture"


def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip(), result.stderr.strip(), result.returncode


sh("docker volume create qdrant_storage")

_, _, exists = sh("docker inspect {name}".format(name=CONTAINER_NAME))
if exists != 0:
    out, err, _ = sh(
        "docker run -d --name {name} -p 6333:6333 -p 6334:6334 "
        "-v qdrant_storage:/qdrant/storage qdrant/qdrant".format(name=CONTAINER_NAME)
    )
else:
    out, err, _ = sh("docker start {name}".format(name=CONTAINER_NAME))
print(out or err)

client = QdrantClient(url="http://localhost:6333")
print(client.get_collections())


e4f50d1f36e8cf40f9df5970752a79ac8ce94081912e24b61a1071b133b9dbad
collections=[CollectionDescription(name='products'), CollectionDescription(name='products_base'), CollectionDescription(name='products_cm'), CollectionDescription(name='terraforming_plans')]


In [3]:
model = SentenceTransformer("intfloat/multilingual-e5-small")
EMBED_DIM = model.get_embedding_dimension()


def embed_passages(texts):
    return model.encode(["passage: " + t for t in texts], normalize_embeddings=True).tolist()


def embed_query(text):
    return model.encode("query: " + text, normalize_embeddings=True).tolist()


def cos(x, y):
    return float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))


with open("books_dataset.json", encoding="utf-8") as f:
    BOOKS = json.load(f)
BOOK_BY_ID = {b["id"]: b for b in BOOKS}
PREFIX = "exercise1_"
BOOKS_COLLECTION = PREFIX + "books"

print("Размерность эмбеддинга:", EMBED_DIM, "| книг в подборке:", len(BOOKS))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 36722.53it/s]


Размерность эмбеддинга: 384 | книг в подборке: 87


## Коллекции и точки

Коллекция `BOOKS_COLLECTION`, которую вы сейчас создадите, останется в Docker volume
`qdrant_storage` и дальше используется без изменений во всех следующих разделах практики —
можно закрывать ноутбук и возвращаться к нему хоть через неделю, данные никуда не денутся.

### Пример — создание коллекции

Синтаксис на игрушечной коллекции (4-мерные векторы, для реальных данных размер будет `EMBED_DIM`):

In [12]:
DEMO_COLLECTION = PREFIX + "demo_scratch"


# collection_exists + delete_collection - стандартный способ сделать создание коллекции идемпотентным
# (можно перезапускать ячейку сколько угодно раз без ошибки "уже существует")
if client.collection_exists(DEMO_COLLECTION):
    client.delete_collection(DEMO_COLLECTION)

client.create_collection(
    collection_name=DEMO_COLLECTION,
    vectors_config=VectorParams(size=4, distance=Distance.COSINE),
)

info = client.get_collection(DEMO_COLLECTION)
print("size:", info.config.params.vectors.size, "| distance:", info.config.params.vectors.distance)

client.delete_collection(DEMO_COLLECTION)  # это была просто демонстрация синтаксиса


size: 4 | distance: Cosine


True

## Задание 1

Создайте настоящую коллекцию `BOOKS_COLLECTION` с одним (безымянным) вектором размера `EMBED_DIM` и метрикой `COSINE`. Если она уже существует — удалите и создайте заново. Она понадобится во всех следующих упражнениях.

**Используйте:** `client.collection_exists(...)`, `client.delete_collection(...)`, `client.create_collection(...)`, `VectorParams(...)`.

In [19]:
BOOKS_COLLECTION = PREFIX + "books"


model = SentenceTransformer("intfloat/multilingual-e5-small")
EMBED_DIM = model.get_embedding_dimension()

if client.collection_exists(BOOKS_COLLECTION):
    client.delete_collection(BOOKS_COLLECTION)

client.create_collection(
    collection_name=BOOKS_COLLECTION,
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE)

)
info = client.get_collection(BOOKS_COLLECTION)
print("size:", info.config.params.vectors.size, "| distance:", info.config.params.vectors.distance)

 

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18580.35it/s]


size: 384 | distance: Cosine


In [20]:
def check_ex1():
    # проверяем и сам факт создания коллекции, и что её конфигурация (размер вектора, метрика) верна
    assert client.collection_exists(BOOKS_COLLECTION), "коллекция {} не найдена".format(BOOKS_COLLECTION)
    info = client.get_collection(BOOKS_COLLECTION)
    vectors_config = info.config.params.vectors
    assert vectors_config.size == EMBED_DIM, "ожидали size={}, получили {}".format(EMBED_DIM, vectors_config.size)
    assert vectors_config.distance == Distance.COSINE, "ожидали COSINE, получили {}".format(vectors_config.distance)
    print("OK: коллекция создана с правильной конфигурацией")
    print("ОТВЕТ ДЛЯ STEPIK:", vectors_config.size)


check_ex1()


OK: коллекция создана с правильной конфигурацией
ОТВЕТ ДЛЯ STEPIK: 384


### Пример — upsert батчами

Прежде чем грузить весь датасет, потренируемся на трёх игрушечных точках — разберём форму `PointStruct` и цикл батчами:

In [21]:
toy_points = [
    PointStruct(id=901, vector=[1.0, 0.0, 0.0, 0.0], payload={"note": "toy-1"}),
    PointStruct(id=902, vector=[0.0, 1.0, 0.0, 0.0], payload={"note": "toy-2"}),
    PointStruct(id=903, vector=[0.0, 0.0, 1.0, 0.0], payload={"note": "toy-3"}),
]

client.create_collection(DEMO_COLLECTION, vectors_config=VectorParams(size=4, distance=Distance.COSINE))

# грузим батчами, а не всё одним upsert - на реальных объёмах данных так меньше нагрузка на один запрос
BATCH = 2
for i in range(0, len(toy_points), BATCH):
    client.upsert(collection_name=DEMO_COLLECTION, points=toy_points[i:i + BATCH])

print("точек в игрушечной коллекции:", client.count(DEMO_COLLECTION).count)
client.delete_collection(DEMO_COLLECTION)


точек в игрушечной коллекции: 3


True

**Альтернативный синтаксис из документации:** `points` можно передать не списком `PointStruct`,
а через `models.Batch(ids=[...], vectors=[...], payloads=[...])` — колоночный формат, три
параллельных списка вместо списка объектов. Это тот же самый `upsert`, просто другая форма записи
одних и тех же данных: `models.Batch` удобен, когда данные уже лежат колонками (например, из
pandas), а список `PointStruct` — когда у вас список объектов по одному на точку, как в этом курсе.

In [38]:
BOOKS

[{'id': 1,
  'title': 'Евгений Онегин',
  'author': 'Александр Пушкин',
  'year': 1833,
  'genre': 'роман',
  'movement': 'романтизм',
  'pages': 200,
  'in_school_curriculum': True,
  'text': 'Столичный денди Евгений Онегин, скучающий и разочарованный светской жизнью, приезжает в деревню, где знакомится с юной Татьяной Лариной. Татьяна влюбляется и пишет ему письмо с признанием, но Онегин отвергает её чувства. На именинах Татьяны Онегин от скуки ухаживает за её сестрой Ольгой, что приводит к дуэли с женихом Ольги поэтом Ленским. Ленский погибает. Спустя годы Онегин встречает Татьяну уже замужней светской дамой и понимает, что любит её, но получает решительный отказ.'},
 {'id': 2,
  'title': 'Герой нашего времени',
  'author': 'Михаил Лермонтов',
  'year': 1840,
  'genre': 'роман',
  'movement': 'романтизм',
  'pages': 220,
  'in_school_curriculum': True,
  'text': 'Офицер Григорий Печорин — умный, циничный и разочарованный в жизни человек, приносящий несчастье всем, кто оказывается ря

In [42]:
df_books= pl.from_dicts(BOOKS)
df_books

id,title,author,year,genre,movement,pages,in_school_curriculum,text
i64,str,str,i64,str,str,i64,bool,str
1,"""Евгений Онегин""","""Александр Пушкин""",1833,"""роман""","""романтизм""",200,true,"""Столичный денди Евгений Онегин…"
2,"""Герой нашего времени""","""Михаил Лермонтов""",1840,"""роман""","""романтизм""",220,true,"""Офицер Григорий Печорин — умны…"
3,"""Мёртвые души""","""Николай Гоголь""",1842,"""роман""","""реализм""",350,true,"""Авантюрист Павел Иванович Чичи…"
4,"""Обломов""","""Иван Гончаров""",1859,"""роман""","""реализм""",480,true,"""Илья Ильич Обломов — добродушн…"
5,"""Обыкновенная история""","""Иван Гончаров""",1847,"""роман""","""реализм""",320,false,"""Восторженный романтичный юноша…"
…,…,…,…,…,…,…,…,…
83,"""Чайка""","""Антон Чехов""",1896,"""пьеса""","""реализм""",75,true,"""Начинающий писатель Константин…"
84,"""Иванов""","""Антон Чехов""",1887,"""пьеса""","""реализм""",80,false,"""Помещик Николай Иванов, некогд…"
85,"""На дне""","""Максим Горький""",1902,"""пьеса""","""реализм""",90,true,"""Обитатели грязной ночлежки — б…"


In [44]:
for elem in df_books:
    print(elem)

shape: (87,)
Series: 'id' [i64]
[
	1
	2
	3
	4
	5
	…
	83
	84
	85
	86
	87
]
shape: (87,)
Series: 'title' [str]
[
	"Евгений Онегин"
	"Герой нашего времени"
	"Мёртвые души"
	"Обломов"
	"Обыкновенная история"
	…
	"Чайка"
	"Иванов"
	"На дне"
	"Живой труп"
	"Свадьба Кречинского"
]
shape: (87,)
Series: 'author' [str]
[
	"Александр Пушкин"
	"Михаил Лермонтов"
	"Николай Гоголь"
	"Иван Гончаров"
	"Иван Гончаров"
	…
	"Антон Чехов"
	"Антон Чехов"
	"Максим Горький"
	"Лев Толстой"
	"Александр Сухово-Кобылин"
]
shape: (87,)
Series: 'year' [i64]
[
	1833
	1840
	1842
	1859
	1847
	…
	1896
	1887
	1902
	1900
	1855
]
shape: (87,)
Series: 'genre' [str]
[
	"роман"
	"роман"
	"роман"
	"роман"
	"роман"
	…
	"пьеса"
	"пьеса"
	"пьеса"
	"пьеса"
	"пьеса"
]
shape: (87,)
Series: 'movement' [str]
[
	"романтизм"
	"романтизм"
	"реализм"
	"реализм"
	"реализм"
	…
	"реализм"
	"реализм"
	"реализм"
	"реализм"
	"реализм"
]
shape: (87,)
Series: 'pages' [i64]
[
	200
	220
	350
	480
	320
	…
	75
	80
	90
	90
	90
]
shape: (87,)
Series:

## Задание 2

Для каждой книги из `BOOKS` постройте `PointStruct`:
- `id` = `book["id"]`;
- `vector` = эмбеддинг **`title + ". " + text`**;
- `payload` = все поля книги, кроме `id`.

Загрузите точки в `BOOKS_COLLECTION` батчами по 16 через `client.upsert`.

**Используйте:** `embed_passages(...)`, `PointStruct(...)`, `client.upsert(...)`.

In [34]:
# ВАШ КОД ЗДЕСЬ
key_to_remove = 'id'   # payload = все поля книги, КРОМЕ id
BATCH = 16

# грузим батчами по BATCH книг за один запрос, а не по одной книге на upsert
for start in range(0, len(BOOKS), BATCH):
    chunk = BOOKS[start:start + BATCH]

    # колоночный формат: ids[n] + vectors[n] + payloads[n] = одна точка n
    ids = [b['id'] for b in chunk]
    texts = [b['title'] + '. ' + b['text'] for b in chunk]
    vectors = embed_passages(texts)   # СТРОКА -> ошибка (обход по символам); список -> список векторов
    # dict comprehension, а не pop: pop удалил бы поле прямо из BOOKS (и из BOOK_BY_ID - там те же объекты)
    payloads = [{k: v for k, v in b.items() if k != key_to_remove} for b in chunk]

    # страховка: три параллельные колонки обязаны совпасть по длине, иначе точки перемешаются
    assert len(ids) == len(payloads) == len(vectors), 'колонки разъехались по длине'

    client.upsert(
        collection_name=BOOKS_COLLECTION,
        points=models.Batch(ids=ids, vectors=vectors, payloads=payloads),
    )

print('точек в коллекции:', client.count(BOOKS_COLLECTION).count)


точек в коллекции: 87


In [35]:
def check_ex2():
    # проверяем и количество точек, и что вектор с payload реально долетели и совпадают с исходными данными
    count = client.count(BOOKS_COLLECTION).count
    assert count == len(BOOKS), "ожидали {} точек, в коллекции {}".format(len(BOOKS), count)

    sample = client.retrieve(BOOKS_COLLECTION, ids=[1], with_vectors=True)[0]
    assert len(sample.vector) == EMBED_DIM, "вектор точки id=1 не той длины"
    assert sample.payload["title"] == BOOK_BY_ID[1]["title"], "payload не совпадает с исходными данными"
    assert "id" not in sample.payload, "id не должен дублироваться внутри payload"

    print("OK: все", count, "точек загружены корректно")
    print("ОТВЕТ ДЛЯ STEPIK:", count)


check_ex2()


OK: все 87 точек загружены корректно
ОТВЕТ ДЛЯ STEPIK: 87


### Пример — retrieve и один scroll

`retrieve` — когда id известны заранее. `scroll` — постраничный обход; один вызов отдаёт только одну страницу и `next_offset` для следующей:

In [ ]:
sample_point = client.retrieve(BOOKS_COLLECTION, ids=[50], with_vectors=False)[0]
print("retrieve:", sample_point.payload["title"])

# scroll без фильтра - просто постраничный обход всей коллекции; next_offset нужен, чтобы запросить следующую страницу
first_page, next_offset = client.scroll(BOOKS_COLLECTION, limit=10, with_payload=False, with_vectors=False)
print("первая страница scroll:", len(first_page), "точек, next_offset =", next_offset)


## Задание 3

Соберите через `scroll` **все** точки коллекции: вызывайте его в цикле, каждый раз передавая
`offset=next_offset` из предыдущего ответа, пока `next_offset` не станет `None`. Сложите все точки
в список `all_scrolled`.

**Используйте:** `client.scroll(...)` в цикле.

In [ ]:
# ВАШ КОД ЗДЕСЬ
all_scrolled = ...


In [ ]:
def check_ex3(all_scrolled):
    assert len(all_scrolled) == len(BOOKS), "scroll собрал {}, ожидали {}".format(len(all_scrolled), len(BOOKS))
    print("OK: scroll дошёл до конца коллекции,", len(all_scrolled), "точек")
    print("ОТВЕТ ДЛЯ STEPIK:", len(all_scrolled))


check_ex3(all_scrolled)


### Пример — set_payload / delete_payload

На книге `id=50` — добавим служебную метку и уберём одно поле, не трогая остальной payload:

In [ ]:
# set_payload/delete_payload точечно правят payload, не трогая вектор и остальные поля точки
client.set_payload(collection_name=BOOKS_COLLECTION, payload={"has_audiobook": True}, points=[50])
client.delete_payload(collection_name=BOOKS_COLLECTION, keys=["movement"], points=[50])

updated = client.retrieve(BOOKS_COLLECTION, ids=[50])[0]
print("has_audiobook:", updated.payload.get("has_audiobook"))
print("movement в payload:", "movement" in updated.payload)
print("title не тронут:", updated.payload["title"])


## Задание 4

У книги `id=86` ("Живой труп" Толстого) недавно вышла новая экранизация. Добавьте точке `id=86` поле `has_film_adaptation=True` через `set_payload`. Затем уберите поле `movement` у этой же точки через `delete_payload`.

**Используйте:** `client.set_payload(...)`, `client.delete_payload(...)`.

In [ ]:
# ВАШ КОД ЗДЕСЬ


In [ ]:
def check_ex4():
    updated = client.retrieve(BOOKS_COLLECTION, ids=[86])[0]
    assert updated.payload.get("has_film_adaptation") is True, "has_film_adaptation должен быть True"
    assert "movement" not in updated.payload, "movement должен быть удалён"
    assert updated.payload["author"] == "Лев Толстой", "остальной payload не должен был измениться"
    print("OK: set_payload и delete_payload применены точечно, остальной payload не тронут")
    print("ОТВЕТ ДЛЯ STEPIK:", len(updated.payload))


check_ex4()
